# 36. Multimodal AI — 이미지와 텍스트를 함께

> **제36장** · **이론편 대응: 24장 (Multimodal AI)**
> **예상 소요**: 70분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: **pillow** (대부분 이미 설치됨)
> **다운로드**: CLIP 약 600MB (자동)

---

## 이 장에서 하는 일

지금까지는 텍스트만 다뤘다. 이번에는 **이미지와 텍스트를 같은 공간에** 놓는다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | **준비 — CLIP 내려받기** | — |
| 2 | 서로 다른 자료를 어떻게 잇는가 | 24.1절 |
| 3 | **CLIP으로 이미지-텍스트 매칭** ★ | 24.2절 |
| 4 | 이미지 검색 | 24.2절 |
| 5 | 제로샷 분류 | 24.2절 |
| 6 | 영상의 계산량 — 이론편 24.4절 검증 | 24.4절 |
| 7 | Vision-Language 모델 | 24.3절 |
| 8 | 한계와 주의점 | 24.5절 |

**3절이 핵심이다.** 27장에서 텍스트끼리 비교했던 코사인 유사도가
이미지와 텍스트 사이에서도 작동하는 것을 확인한다.

---

## 1. 준비 — CLIP 내려받기

### CLIP이란

**Contrastive Language-Image Pre-training** — 이미지와 텍스트를 **같은 벡터 공간**에 놓도록
학습된 모델이다. 이론편 24.2절에서 다룬 구조다.

| 항목 | 내용 |
|---|---|
| 모델 | `openai/clip-vit-base-patch32` |
| 크기 | 약 600MB |
| 임베딩 차원 | 512 |
| 구성 | 이미지 인코더 + 텍스트 인코더 |

### 필요한 패키지

```
pip install pillow
```

`transformers`(23장)와 `torch`는 이미 설치되어 있다.
`pillow`는 이미지를 다루는 라이브러리로, 대부분 이미 깔려 있다.

In [ ]:
import importlib

print("=" * 60)
print("필요 패키지 확인")
print("=" * 60)

required = [
    ("PIL", "이미지 처리 (pillow)", "pip install pillow"),
    ("transformers", "CLIP 모델 (23장)", "pip install transformers"),
    ("torch", "PyTorch", "pip install torch"),
    ("numpy", "수치 계산", "pip install numpy"),
]

missing = []
for name, desc, install in required:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, "__version__", "설치됨")
        print(f"[OK]   {name:<16}{ver:<14}{desc}")
    except ImportError:
        print(f"[없음] {name:<16}{'':<14}{desc}")
        missing.append(install)

print("-" * 60)
if missing:
    print("설치가 필요합니다:")
    for cmd in set(missing):
        print(f"  {cmd}")
else:
    print("[준비 완료] 2절로 진행하세요.")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
import time
from PIL import Image

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} / 장치: {device}")

from transformers import CLIPModel, CLIPProcessor

CLIP_NAME = "openai/clip-vit-base-patch32"
print(f"\nCLIP 불러오는 중: {CLIP_NAME}")
print("처음 실행하면 약 600MB를 내려받습니다.")

t0 = time.time()
clip_model = CLIPModel.from_pretrained(CLIP_NAME).to(device)
clip_processor = CLIPProcessor.from_pretrained(CLIP_NAME)
clip_model.eval()
print(f"로드 완료: {time.time()-t0:.1f}초")
print()
print(f"파라미터    : {sum(p.numel() for p in clip_model.parameters())/1e6:.0f}M")
print(f"임베딩 차원 : {clip_model.config.projection_dim}")

---

## 2. 서로 다른 자료를 어떻게 잇는가 — 이론편 24.1절

이미지는 픽셀 배열이고 텍스트는 토큰 열이다. **형태가 완전히 다르다.**

이론편 24.1절에서 다룬 해법은 이렇다.

> **둘을 같은 벡터 공간으로 보낸다.**

각각 다른 인코더를 쓰되, **출력 차원을 맞추고** "짝이 맞는 이미지-텍스트는 가깝게,
아닌 것은 멀게" 학습시킨다.

```
이미지 → 이미지 인코더(ViT) ─┐
                             ├→ 같은 512차원 공간
텍스트 → 텍스트 인코더 ───────┘
```

그러면 **27장에서 텍스트끼리 했던 코사인 유사도 계산**을 이미지-텍스트 사이에서도 할 수 있다.

In [ ]:
import torch

print("=" * 70)
print("CLIP의 두 인코더")
print("=" * 70)

vision = clip_model.vision_model
text = clip_model.text_model

n_vision = sum(p.numel() for p in vision.parameters())
n_text = sum(p.numel() for p in text.parameters())

print(f"{'구성':<20}{'파라미터':<16}{'구조'}")
print("-" * 70)
print(f"{'이미지 인코더':<20}{n_vision/1e6:<16.1f}M  ViT (16장의 CNN 대신 Transformer)")
print(f"{'텍스트 인코더':<20}{n_text/1e6:<16.1f}M  Transformer (17~18번 구조)")
print("-" * 70)
print()
print("두 인코더 모두 Transformer 계열이다.")
print("  이미지도 패치로 잘라 토큰처럼 다룬다 (Vision Transformer)")
print()

# 이미지 패치 구조 확인
patch_size = clip_model.config.vision_config.patch_size
image_size = clip_model.config.vision_config.image_size
n_patches = (image_size // patch_size) ** 2

print(f"이미지 처리 방식")
print(f"  입력 크기 : {image_size} x {image_size}")
print(f"  패치 크기 : {patch_size} x {patch_size}")
print(f"  패치 개수 : ({image_size}//{patch_size})^2 = {n_patches}")
print()
print(f"이미지 한 장이 {n_patches}개 토큰이 된다 — 6절에서 이 수치가 중요해진다.")

In [ ]:
import numpy as np
from PIL import Image


def make_shape_image(shape, color, size=224):
    """실습용 도형 이미지를 만든다 (외부 다운로드 없이)"""
    arr = np.ones((size, size, 3), dtype=np.uint8) * 245

    if shape == "square":
        arr[60:164, 60:164] = color
    elif shape == "circle":
        y, x = np.ogrid[:size, :size]
        mask = (x - size//2)**2 + (y - size//2)**2 <= 52**2
        arr[mask] = color
    elif shape == "triangle":
        for i in range(104):
            half = i // 2
            arr[60+i, size//2-half:size//2+half] = color
    elif shape == "stripes":
        for i in range(0, size, 24):
            arr[i:i+12] = color

    return Image.fromarray(arr)


COLORS = {"red": (220, 40, 40), "blue": (40, 80, 220),
          "green": (40, 180, 80), "yellow": (240, 200, 40)}

images = {
    "빨간 사각형": make_shape_image("square", COLORS["red"]),
    "파란 원":     make_shape_image("circle", COLORS["blue"]),
    "초록 삼각형": make_shape_image("triangle", COLORS["green"]),
    "노란 줄무늬": make_shape_image("stripes", COLORS["yellow"]),
}

print("=" * 60)
print("실습용 이미지 생성")
print("=" * 60)
print(f"{len(images)}장을 코드로 만들었다 (외부 다운로드 없음)")
print()

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(images), figsize=(12, 3.2))
for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(img)
    ax.set_title(name, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

print("실제 사진 대신 도형을 쓰는 이유")
print("  정답이 명확해 결과를 판단하기 쉽다.")
print("  저작권 문제 없이 누구나 같은 실험을 할 수 있다.")

---

## 3. 이미지-텍스트 매칭 ★ — 이론편 24.2절

이제 **이미지와 텍스트의 유사도**를 계산한다.
27장에서 텍스트끼리 했던 것과 같은 방식이다.

In [ ]:
import torch


def get_image_emb(images):
    """이미지 임베딩을 꺼낸다.

    transformers 버전에 따라 get_image_features 의 반환 형태가 다르다.
    - 구버전: 텐서를 바로 반환
    - 신버전: pooler_output 을 담은 객체를 반환
    둘 다 처리한다.
    """
    inputs = clip_processor(images=images, return_tensors="pt").to(device)
    with torch.no_grad():
        out = clip_model.get_image_features(**inputs)
    return out if hasattr(out, "shape") else out.pooler_output


def get_text_emb(texts):
    """텍스트 임베딩을 꺼낸다 (위와 같은 이유로 분기)"""
    inputs = clip_processor(text=texts, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        out = clip_model.get_text_features(**inputs)
    return out if hasattr(out, "shape") else out.pooler_output


print("임베딩 추출 함수 준비 완료")
print("  transformers 버전 차이를 흡수하도록 만들었다.")

In [ ]:
import torch
import numpy as np

texts = [
    "a red square",
    "a blue circle",
    "a green triangle",
    "yellow stripes",
    "a photo of a cat",     # 관련 없는 것
]

image_list = list(images.values())
image_names = list(images.keys())

print("=" * 78)
print("이미지-텍스트 매칭")
print("=" * 78)

inputs = clip_processor(text=texts, images=image_list,
                        return_tensors="pt", padding=True).to(device)

with torch.no_grad():
    outputs = clip_model(**inputs)

# logits_per_image: 각 이미지가 각 텍스트와 얼마나 맞는가
probs = outputs.logits_per_image.softmax(dim=-1).cpu().numpy()

print(f"{'이미지':<16}" + "".join(f"{t[:13]:>15}" for t in texts))
print("-" * 78)
for i, name in enumerate(image_names):
    row = "".join(f"{probs[i,j]:>15.4f}" for j in range(len(texts)))
    print(f"{name:<16}{row}")
print("-" * 78)
print()

# 대각선이 가장 큰지 확인
correct = sum(int(probs[i].argmax() == i) for i in range(len(image_names)))
print(f"정확히 맞춘 이미지: {correct}/{len(image_names)}")
print()
for i, name in enumerate(image_names):
    best = probs[i].argmax()
    print(f"  {name:<14}→ '{texts[best]}' ({probs[i][best]:.4f})")
print()
print("대각선 값이 가장 크다 — 각 이미지가 맞는 설명을 골랐다.")
print("'a photo of a cat' 은 어느 이미지에서도 낮다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(probs, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label="확률")

for i in range(len(image_names)):
    for j in range(len(texts)):
        color = "white" if probs[i,j] > 0.5 else "black"
        ax.text(j, i, f"{probs[i,j]:.3f}", ha="center", va="center",
                fontsize=9, color=color)

ax.set_xticks(range(len(texts)))
ax.set_xticklabels(texts, rotation=30, ha="right", fontsize=9)
ax.set_yticks(range(len(image_names)))
ax.set_yticklabels(image_names, fontsize=9)
ax.set_title("이미지-텍스트 매칭 확률")
plt.tight_layout()
plt.show()

print("27장 8절의 Attention 히트맵과 읽는 법이 같다.")
print("  대각선이 밝으면 짝이 잘 맞은 것이다.")

In [ ]:
import torch
import numpy as np

print("=" * 70)
print("임베딩을 직접 꺼내 코사인 유사도 계산 (22번 방식)")
print("=" * 70)

img_emb = get_image_emb(image_list)
txt_emb = get_text_emb(texts)

print(f"이미지 임베딩: {tuple(img_emb.shape)}")
print(f"텍스트 임베딩: {tuple(txt_emb.shape)}")
print(f"  → 같은 512차원 공간에 있다")
print()

# 27장 3절에서 배운 대로 정규화 후 내적
img_norm = img_emb / img_emb.norm(dim=-1, keepdim=True)
txt_norm = txt_emb / txt_emb.norm(dim=-1, keepdim=True)
cosine = (img_norm @ txt_norm.T).cpu().numpy()

print("코사인 유사도 (정규화 후 내적)")
print(f"{'이미지':<16}" + "".join(f"{t[:13]:>15}" for t in texts))
print("-" * 78)
for i, name in enumerate(image_names):
    row = "".join(f"{cosine[i,j]:>15.4f}" for j in range(len(texts)))
    print(f"{name:<16}{row}")
print("-" * 78)
print()
print("확률(softmax)과 달리 원본 유사도 값이다.")
print("  범위가 대체로 0.1~0.4 정도로 좁다 — CLIP의 특성이다.")
print("  절대값보다 **상대적 순위**를 보는 것이 중요하다.")

### 왜 유사도 범위가 좁은가

코사인 유사도가 0.15~0.35 정도에 몰려 있다. 27장의 텍스트 임베딩과 다르다.

**CLIP은 대조 학습(contrastive learning)으로 훈련되었기 때문**이다.
"맞는 짝을 다른 것보다 높게"만 학습했지, 절대값을 특정 범위로 맞추지는 않았다.

따라서 **0.3이 높은 값인지 낮은 값인지는 다른 후보와 비교해야 안다.**
그래서 실무에서는 소프트맥스를 씌워 상대 확률로 보는 경우가 많다.

---

## 4. 이미지 검색 — 이론편 24.2절

27장의 벡터 검색을 이미지에 적용한다. **텍스트로 이미지를 찾는 것**이다.

In [ ]:
import torch
import numpy as np


class ImageSearcher:
    # 텍스트로 이미지를 찾는 검색기 (27장 6절의 이미지 버전)

    def __init__(self, model, processor, device):
        self.model = model
        self.processor = processor
        self.device = device
        self.images = []
        self.names = []
        self.embeddings = None

    def add(self, images_dict):
        """이미지를 벡터로 바꿔 저장"""
        names = list(images_dict.keys())
        imgs = list(images_dict.values())

        emb = get_image_emb(imgs)
        emb = emb / emb.norm(dim=-1, keepdim=True)      # 정규화 (27장 3절)

        self.images.extend(imgs)
        self.names.extend(names)
        self.embeddings = emb if self.embeddings is None else torch.cat([self.embeddings, emb])

    def search(self, query, top_k=3):
        """텍스트로 이미지 검색"""
        q_emb = get_text_emb([query])
        q_emb = q_emb / q_emb.norm(dim=-1, keepdim=True)

        scores = (self.embeddings @ q_emb.T).squeeze(-1).cpu().numpy()
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [(self.names[i], float(scores[i]), self.images[i]) for i in top_idx]


# 이미지를 더 만들어 검색 대상을 늘린다
more_images = dict(images)
more_images.update({
    "빨간 원":     make_shape_image("circle", COLORS["red"]),
    "파란 사각형": make_shape_image("square", COLORS["blue"]),
    "초록 줄무늬": make_shape_image("stripes", COLORS["green"]),
})

searcher = ImageSearcher(clip_model, clip_processor, device)
searcher.add(more_images)

print("=" * 70)
print("이미지 검색")
print("=" * 70)
print(f"저장된 이미지: {len(searcher.names)}장")
print()

queries = ["something red", "a circle", "stripes pattern"]

for q in queries:
    print(f"\n질문: '{q}'")
    for rank, (name, score, _) in enumerate(searcher.search(q, top_k=3), 1):
        print(f"  {rank}. [{score:.4f}] {name}")

In [ ]:
import matplotlib.pyplot as plt

query = "something red"
results = searcher.search(query, top_k=3)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, (name, score, img) in zip(axes, results):
    ax.imshow(img)
    ax.set_title(f"{name}\n{score:.4f}", fontsize=10)
    ax.axis("off")

fig.suptitle(f"검색: '{query}'", fontsize=12)
plt.tight_layout()
plt.show()

print("색·모양 같은 시각적 속성으로 검색이 된다.")
print()
print("주목할 점: 'red' 라는 단어가 이미지 파일명에 없어도 찾는다.")
print("  이미지의 내용을 이해해 매칭하는 것이다 (이론편 24.2절).")

---

## 5. 제로샷 분류 — 이론편 24.2절

CLIP의 흥미로운 성질이 있다. **학습하지 않은 분류 작업을 바로 할 수 있다.**

07장에서는 분류기를 학습시켜야 했다. CLIP은 **분류 대상을 문장으로 적어 주기만** 하면 된다.

```python
labels = ["a red object", "a blue object", "a green object"]
# 이 중 어느 것과 가장 가까운가?
```

이를 제로샷(zero-shot) 분류라 한다.

In [ ]:
import torch
import numpy as np


def zero_shot_classify(image, class_names, template="a photo of {}"):
    """학습 없이 분류 (이론편 24.2절)

    template: 클래스 이름을 문장으로 감싸는 형식
              CLIP은 문장으로 학습되었으므로 단어만 주는 것보다 낫다
    """
    prompts = [template.format(c) for c in class_names]

    inputs = clip_processor(text=prompts, images=[image],
                            return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = clip_model(**inputs)

    probs = outputs.logits_per_image.softmax(dim=-1)[0].cpu().numpy()
    return dict(zip(class_names, probs))


print("=" * 70)
print("제로샷 분류 — 색 맞히기")
print("=" * 70)

color_classes = ["red", "blue", "green", "yellow"]

print(f"{'이미지':<16}" + "".join(f"{c:>12}" for c in color_classes) + "   예측")
print("-" * 70)
for name, img in images.items():
    result = zero_shot_classify(img, color_classes, "a photo of a {} shape")
    row = "".join(f"{result[c]:>12.3f}" for c in color_classes)
    pred = max(result, key=result.get)
    print(f"{name:<16}{row}   {pred}")
print("-" * 70)
print()
print("색 이름을 학습시킨 적이 없는데도 구분한다.")

In [ ]:
print("=" * 70)
print("같은 이미지, 다른 분류 기준")
print("=" * 70)

img = images["파란 원"]

tasks = {
    "색":     (["red", "blue", "green", "yellow"], "a photo of a {} shape"),
    "모양":   (["square", "circle", "triangle", "stripes"], "a photo of a {}"),
    "종류":   (["a geometric shape", "an animal", "a landscape", "text"], "{}"),
}

for task_name, (classes, template) in tasks.items():
    result = zero_shot_classify(img, classes, template)
    pred = max(result, key=result.get)
    print(f"\n[{task_name} 분류]")
    for c, p in sorted(result.items(), key=lambda x: -x[1]):
        bar = "█" * int(p * 30)
        print(f"  {c:<22}{p:.4f}  {bar}")
    print(f"  → 예측: {pred}")

print()
print("=" * 70)
print("같은 이미지를 세 가지 기준으로 분류했다.")
print()
print("07장의 분류기와 무엇이 다른가")
print("  06번: 분류 대상이 고정 (학습 시 정한 클래스만)")
print("  CLIP: 문장만 바꾸면 새 분류 작업이 된다")
print()
print("이것이 이론편 19.4절에서 다룬 Foundation Model 의 성질이다.")

In [ ]:
print("=" * 70)
print("프롬프트 템플릿의 영향")
print("=" * 70)
print()
print("CLIP은 '문장'으로 학습되었으므로 어떻게 표현하느냐가 영향을 준다.")
print()

img = images["빨간 사각형"]
classes = ["red", "blue", "green", "yellow"]

templates = [
    "{}",
    "a {} object",
    "a photo of a {} shape",
    "an image showing the color {}",
]

print(f"{'템플릿':<34}{'red':<10}{'blue':<10}{'green':<10}{'yellow'}")
print("-" * 70)
for t in templates:
    r = zero_shot_classify(img, classes, t)
    row = "".join(f"{r[c]:<10.3f}" for c in classes)
    print(f"{t:<34}{row}")
print("-" * 70)
print()
print("템플릿에 따라 확률이 달라진다.")
print()
print("실무 요령")
print("  - 여러 템플릿의 결과를 평균 내면 안정적이다 (prompt ensembling)")
print("  - 어떤 표현이 잘 되는지는 실험으로 찾는다")
print("  - 24장에서 다룬 프롬프트 설계와 같은 문제다")

---

## 6. 영상의 계산량 — 이론편 24.4절 검증 ★

이론편 24.4절에서 **영상 1분이 만드는 토큰 수**를 계산했다. 그 값을 확인한다.

224×224 이미지를 16×16 패치로 나누면

$$N_{patch} = \left(\frac{224}{16}\right)^2 = 196$$

**이론편에서 계산한 표**

| 영상 | 프레임 | 총 토큰 | Attention 계산량 |
|---|---|---|---|
| 10초, 1fps | 10 | 1,960 | 1배 |
| 60초, 1fps | 60 | 11,760 | 36배 |
| 60초, 8fps | 480 | **94,080** | **2,300배** |

In [ ]:
import numpy as np

print("=" * 78)
print("이론편 24.4절 값 검증")
print("=" * 78)

IMG_SIZE = 224
PATCH = 16
n_patch = (IMG_SIZE // PATCH) ** 2

print(f"이미지 {IMG_SIZE}x{IMG_SIZE}, 패치 {PATCH}x{PATCH}")
print(f"  ({IMG_SIZE}//{PATCH})^2 = {IMG_SIZE//PATCH}^2 = {n_patch} 패치")
print()
assert n_patch == 196
print("[OK] 이론편과 일치 (196 패치)")
print()

cases = [
    (10, 1, 1960),
    (60, 1, 11760),
    (60, 8, 94080),
]

print(f"{'영상':<18}{'프레임':<12}{'총 토큰':<16}{'이론편':<14}{'계산량(상대)'}")
print("-" * 78)
base_tokens = None
for sec, fps, book_tokens in cases:
    frames = sec * fps
    tokens = frames * n_patch
    if base_tokens is None:
        base_tokens = tokens
    ratio = (tokens / base_tokens) ** 2      # Attention은 토큰 수의 제곱
    print(f"{f'{sec}초, {fps}fps':<18}{frames:<12}{tokens:<16,}{book_tokens:<14,}{ratio:>10.0f}배")
    assert tokens == book_tokens, f"{sec}초 {fps}fps 토큰 수가 이론편과 다릅니다"

print("-" * 78)
print("[OK] 이론편 24.4절 표와 일치")
print()
print("이론편 18.1절에서 봤듯 Self-Attention 계산량은 토큰 수의 **제곱**에 비례한다.")
print(f"  토큰이 {94080/1960:.0f}배 늘면 계산은 {(94080/1960)**2:.0f}배가 된다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# --- 왼쪽: fps에 따른 토큰 수 ---
ax = axes[0]
fps_range = np.array([1, 2, 4, 8, 16, 30])
for sec, color in [(10, "#0D9488"), (30, "#EA580C"), (60, "#DC2626")]:
    tokens = sec * fps_range * n_patch
    ax.plot(fps_range, tokens, marker="o", linewidth=2,
            color=color, label=f"{sec}초")
ax.axhline(128000, color="gray", linestyle="--", linewidth=1.5)
ax.text(2, 145000, "문맥 창 예시 (12.8만)", fontsize=8, color="gray")
ax.set_xlabel("초당 프레임 (fps)")
ax.set_ylabel("총 토큰 수")
ax.set_yscale("log")
ax.set_title("영상 길이와 fps에 따른 토큰")
ax.legend()
ax.grid(alpha=0.3, which="both")

# --- 오른쪽: 계산량 ---
ax = axes[1]
labels = ["10초\n1fps", "60초\n1fps", "60초\n8fps"]
tokens_list = [1960, 11760, 94080]
compute = [(t/1960)**2 for t in tokens_list]
bars = ax.bar(range(3), compute, color=["#0D9488", "#EA580C", "#DC2626"])
for b, v in zip(bars, compute):
    ax.text(b.get_x()+b.get_width()/2, v*1.15, f"{v:.0f}배",
            ha="center", fontsize=10)
ax.set_yscale("log")
ax.set_xticks(range(3))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("Attention 계산량 (상대)")
ax.set_title("계산량은 토큰 수의 제곱")
ax.grid(axis="y", alpha=0.3, which="both")

plt.tight_layout()
plt.show()

print("실무에서 쓰는 타협 (이론편 24.4절)")
print("  1) 프레임을 듬성듬성 뽑는다 (초당 1프레임 이하)")
print("  2) 패치를 크게 잡는다 (32x32 이면 토큰이 1/4)")
print("  3) 희소 Attention 을 쓴다 (이론편 18.7절)")
print("  어느 쪽이든 정보를 일부 버리는 대가를 치른다.")

---

## 7. Vision-Language 모델 — 이론편 24.3절

CLIP은 이미지와 텍스트를 **매칭**할 수 있지만, **문장을 생성하지는 못한다.**

이미지를 보고 설명을 쓰거나 질문에 답하려면 **VLM(Vision-Language Model)**이 필요하다.

```
이미지 → 비전 인코더 → 투영층 → LLM → 텍스트 생성
                       (차원 맞춤)
```

**핵심은 투영층**이다. 비전 인코더의 출력을 LLM이 이해할 수 있는 형태로 바꿔 주는 부분이다.
LLM 입장에서는 "이미지에서 온 특별한 토큰들"이 앞에 붙은 것처럼 보인다.

In [ ]:
print("=" * 78)
print("CLIP vs VLM (이론편 24.2~24.3절)")
print("=" * 78)
print()
print(f"{'항목':<20}{'CLIP':<28}{'VLM'}")
print("-" * 78)
rows = [
    ("하는 일", "이미지-텍스트 매칭", "이미지 보고 문장 생성"),
    ("출력", "유사도 점수", "자유로운 텍스트"),
    ("구성", "인코더 2개", "비전 인코더 + LLM"),
    ("가능한 작업", "검색, 제로샷 분류", "캡셔닝, VQA, OCR, 추론"),
    ("크기", "수백 MB", "수 GB 이상"),
]
for a, b, c in rows:
    print(f"{a:<20}{b:<28}{c}")
print("-" * 78)
print()
print("VLM 이 할 수 있는 것")
print("  - 이미지 캡셔닝: '해변에서 개가 뛰고 있다'")
print("  - VQA: '사진 속 사람은 몇 명인가요?'")
print("  - OCR: 이미지 속 글자 읽기")
print("  - 도표 이해: 그래프를 보고 추세 설명")
print()
print("이 장에서 VLM 을 직접 돌리지 않는 이유")
print("  작은 VLM 도 수 GB 이며, CPU 에서는 매우 느리다.")
print("  8GB GPU 라면 3B급 VLM 을 INT4 로 시도해 볼 수 있다 (32장 9절).")

In [ ]:
print("=" * 70)
print("VLM 사용 코드 형태")
print("=" * 70)
print()
print("[방법 1] API 사용 (25장 방식)")
print()
code_api = [
    "import base64",
    "",
    "with open('image.jpg', 'rb') as f:",
    "    b64 = base64.b64encode(f.read()).decode()",
    "",
    "response = client.chat.completions.create(",
    "    model='<멀티모달 지원 모델>',",
    "    messages=[{",
    "        'role': 'user',",
    "        'content': [",
    "            {'type': 'text', 'text': '이 이미지를 설명해주세요.'},",
    "            {'type': 'image_url',",
    "             'image_url': {'url': f'data:image/jpeg;base64,{b64}'}},",
    "        ],",
    "    }],",
    ")",
]
for line in code_api:
    print("  " + line)

print()
print("[방법 2] 로컬 모델 (transformers)")
print()
code_local = [
    "from transformers import AutoProcessor, AutoModelForVision2Seq",
    "from PIL import Image",
    "",
    "processor = AutoProcessor.from_pretrained('<VLM 모델명>')",
    "model = AutoModelForVision2Seq.from_pretrained('<VLM 모델명>')",
    "",
    "image = Image.open('photo.jpg')",
    "prompt = '이 사진에 무엇이 보이나요?'",
    "",
    "inputs = processor(images=image, text=prompt, return_tensors='pt')",
    "output = model.generate(**inputs, max_new_tokens=100)",
    "print(processor.decode(output[0], skip_special_tokens=True))",
]
for line in code_local:
    print("  " + line)

print()
print("-" * 70)
print("주의")
print("  메시지 형식이 24장의 ChatML 과 다르다.")
print("  content 가 문자열이 아니라 **리스트**이며, 텍스트와 이미지가 섞인다.")
print("  모델마다 형식이 다르므로 문서 확인이 필요하다.")

---

## 8. 한계와 주의점 — 이론편 24.5절

멀티모달 모델을 쓸 때 알아 둘 것들이다.

In [ ]:
import torch
import numpy as np

print("=" * 70)
print("CLIP이 어려워하는 것")
print("=" * 70)
print()

# 개수 세기 테스트
def make_multi_circles(n, size=224):
    arr = np.ones((size, size, 3), dtype=np.uint8) * 245
    positions = [(60, 60), (60, 160), (160, 60), (160, 160), (110, 110)]
    for i in range(min(n, len(positions))):
        cy, cx = positions[i]
        y, x = np.ogrid[:size, :size]
        mask = (x - cx)**2 + (y - cy)**2 <= 28**2
        arr[mask] = COLORS["blue"]
    return Image.fromarray(arr)


count_images = {f"원 {n}개": make_multi_circles(n) for n in [1, 2, 3, 4]}
count_classes = ["one circle", "two circles", "three circles", "four circles"]

print("[개수 세기]")
print(f"{'이미지':<12}" + "".join(f"{c[:10]:>13}" for c in count_classes) + "   예측")
print("-" * 70)
correct = 0
for i, (name, img) in enumerate(count_images.items()):
    r = zero_shot_classify(img, count_classes, "a photo of {}")
    row = "".join(f"{r[c]:>13.3f}" for c in count_classes)
    pred = max(r, key=r.get)
    ok = count_classes.index(pred) == i
    correct += ok
    print(f"{name:<12}{row}   {pred[:12]}")
print("-" * 70)
print(f"정확히 맞춘 것: {correct}/{len(count_images)}")
print()
print("개수를 정확히 세는 것은 CLIP이 잘 못하는 일이다.")
print("  '무엇이 있는가'는 잘 알지만 '몇 개인가'는 약하다 (이론편 24.5절).")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, (name, img) in zip(axes, count_images.items()):
    ax.imshow(img)
    ax.set_title(name, fontsize=10)
    ax.axis("off")
fig.suptitle("개수 세기 테스트", fontsize=12)
plt.tight_layout()
plt.show()

print("=" * 78)
print("멀티모달 모델의 알려진 한계 (이론편 24.5절)")
print("=" * 78)
print()
print(f"{'한계':<20}{'설명':<34}{'대응'}")
print("-" * 78)
limits = [
    ("개수 세기",       "정확한 수량 파악이 약함",       "필요하면 별도 도구 사용"),
    ("공간 관계",       "'왼쪽/오른쪽' 구분이 불안정",   "프롬프트로 명시"),
    ("작은 글자",       "저해상도에서 OCR 실패",        "확대 후 처리"),
    ("세밀한 구분",     "비슷한 품종·모델 구분 어려움",  "전용 모델 사용"),
    ("학습 데이터 편향", "특정 문화권·인종에 치우침",     "결과 검토 필요"),
    ("환각",           "없는 것을 있다고 설명",        "28장의 검증 방식 적용"),
]
for a, b, c in limits:
    print(f"{a:<20}{b:<34}{c}")
print("-" * 78)
print()
print("[가장 주의할 것] 학습 데이터 편향")
print()
print("  CLIP은 인터넷 이미지-텍스트 쌍으로 학습되었다.")
print("  그 데이터에 담긴 편견이 그대로 반영될 수 있다.")
print("  사람과 관련된 판단에 쓸 때는 특히 신중해야 한다 (이론편 24.5절).")

---

## 9. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| 24.2 | 이미지-텍스트 같은 공간 | 코사인 유사도 계산 ✓ |
| 24.2 | 제로샷 분류 | 학습 없이 분류 ✓ |
| **24.4** | **영상 토큰 1,960 / 11,760 / 94,080** | **일치** ✓ |
| 24.4 | 계산량은 토큰 수의 제곱 | 2,300배 확인 ✓ |
| 24.5 | 개수 세기의 어려움 | 실험 확인 ✓ |

### 27장과 이어지는 지점

| 22번 (텍스트) | 28번 (이미지+텍스트) |
|---|---|
| 텍스트 → 벡터 | 이미지·텍스트 → **같은** 벡터 공간 |
| 코사인 유사도 | 동일 |
| 정규화 후 내적 | 동일 |
| 텍스트 검색 | **텍스트로 이미지 검색** |

**임베딩의 개념이 그대로 확장된다.** 22번을 이해했다면 이 장은 자연스럽다.

### 기억할 것

| 항목 | 요점 |
|---|---|
| CLIP | 인코더 2개 — 매칭은 되지만 생성은 불가 |
| 유사도 범위 | 좁음 — **상대 순위**로 판단 |
| 제로샷 분류 | 문장만 바꾸면 새 작업 |
| 프롬프트 템플릿 | 표현에 따라 결과가 달라짐 |
| 이미지 토큰 | 224x224 → 196 토큰 |
| 영상 | 토큰이 폭증 — 프레임 샘플링 필수 |
| VLM | 비전 인코더 + 투영층 + LLM |
| 편향 | 사람 관련 판단에 신중 |

### 다음 장

**37. AI Agent — 스스로 판단하고 행동하기** — 이론편 32장.
지금까지 만든 것들(RAG·도구·추론)을 엮어 **스스로 판단하고 행동하는** 시스템을 만든다.
24장 7절의 구조화된 출력이 여기서 도구 호출로 이어진다.